# 01 — FastSurfer vs T1Prep concordance

On DLBS sub-1003 (all three waves), compute:

- Brain-mask Dice between FastSurfer `mri/mask.mgz` and T1Prep `mri/p0*.nii.gz` (>0 binarised).
- DK cortical ROI volume correlation (FastSurfer `aseg+DKT.VINN.stats` vs T1Prep ROI volumes derived from the DK40 `.annot` + tissue probability maps).
- Bethlehem 5 (GMV / WMV / sGMV / VentCSF / TCV) concordance — both tools can emit them, with different definitions.
- Within-subject longitudinal stability across three waves.

This notebook assumes outputs under `/data/datasets/smri-fm-cmp/{fastsurfer,t1prep}/ds004856/sub-1003_ses-wave{1,2,3}/`.

In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

ROOT = Path('/data/datasets/smri-fm-cmp')
FS = ROOT / 'fastsurfer' / 'ds004856'
T1P = ROOT / 't1prep' / 'ds004856'
SUB = 'sub-1003'
WAVES = ['ses-wave1', 'ses-wave2', 'ses-wave3']

## Brain-mask Dice

FastSurfer `mri/mask.mgz` is the conformed 256^3 brain mask at 1 mm. T1Prep's
`mri/p0*.nii.gz` is a tissue-class labelmap in native space; binarise where label > 0.
Resample to a common grid before Dice.

In [ ]:
def dice(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool)
    inter = (a & b).sum()
    denom = a.sum() + b.sum()
    return float(2 * inter / denom) if denom else float('nan')

# TODO: resample both masks to the same grid (choose T1Prep native as reference)
# using nibabel.processing.resample_from_to, then compute Dice per wave.

## DK ROI volume correlation

FastSurfer's `stats/aseg+DKT.VINN.stats` gives `ctx-lh-<region>` + `ctx-rh-<region>` 
volume in mm^3 for 34×2 regions. T1Prep's DK40 annotations plus the native-space tissue
probability maps let us reconstruct the same table — sum GM voxels inside each vertex-mapped
ROI projected to volume space via the inverse of the pial/white-derived ribbon.

In [ ]:
def parse_fastsurfer_stats(path: Path) -> pd.DataFrame:
    rows = []
    for line in path.read_text().splitlines():
        if line.startswith('#') or not line.strip():
            continue
        parts = line.split()
        # columns: Index SegId NVoxels Volume_mm3 StructName normMean normStdDev normMin normMax normRange
        rows.append({
            'struct': parts[4],
            'volume_mm3': float(parts[3]),
            'n_voxels': int(parts[2]),
        })
    return pd.DataFrame(rows)

fs_rows = {}
for w in WAVES:
    p = FS / f'{SUB}_{w}' / 'stats' / 'aseg+DKT.VINN.stats'
    if p.exists():
        fs_rows[w] = parse_fastsurfer_stats(p)
        print(w, len(fs_rows[w]), 'regions')

## Bethlehem 5 headline metrics

Following `smri-fm/preprocessing/pipeline.py::parse_synthseg_volumes`, reconstruct
GMV / WMV / sGMV / VentCSF / TCV from each tool's outputs and scatter FastSurfer
vs T1Prep per wave.

In [ ]:
GMV_COLS = ('left cerebral cortex', 'right cerebral cortex')
WMV_COLS = ('left cerebral white matter', 'right cerebral white matter')
SGMV_COLS = (
    'left thalamus', 'right thalamus',
    'left caudate', 'right caudate',
    'left putamen', 'right putamen',
    'left pallidum', 'right pallidum',
    'left hippocampus', 'right hippocampus',
    'left amygdala', 'right amygdala',
    'left accumbens area', 'right accumbens area',
)
VENTCSF_COLS = (
    'left lateral ventricle', 'right lateral ventricle',
    'left inferior lateral ventricle', 'right inferior lateral ventricle',
    '3rd ventricle', '4th ventricle',
)
# TODO: compute Bethlehem 5 per tool per wave, then scatter.

## Longitudinal stability (three waves, one subject)

sub-1003 is scanned at age 54, 58, 63 (~9 yr span, MMSE 28 / 30 / 28 — cognitively stable).
Expectation: GMV decreases linearly; ventricles enlarge; cortical thickness in DK ROIs
drifts. Deviations between FastSurfer and T1Prep here tell us whether the tools are
*longitudinally consistent* — a separate property from cross-sectional agreement.